# 3. LRU Cache
**Difficulty:** 🟡 Medium · **Topic:** Hash Maps + Doubly Linked List · **LeetCode:** https://leetcode.com/problems/lru-cache/

> **DevRev context:** cache the latest **task states**, **API responses**, or **user sessions** with a fixed memory budget. When it fills up, evict the **Least Recently Used** entry. Both `get` and `put` must be **O(1)** — a cache that's O(n) per access defeats the purpose. This is a top senior-level design question.

## 💡 Concepts

**Core concept(s):** Combine a **hash map** (instant lookup by key) with a **doubly linked list** (instant reorder / eviction).

**Why it applies here:** You need two things fast: find a value by key (hash map), and know which entry is least-recently-used so you can evict it (ordering). A doubly linked list keeps entries in use-order and lets you move a node to the front or drop the back node in O(1). Neither structure alone does both cheaply — together they do.

**Key intuition:** Most-recently-used sits at the front; the least-recently-used falls off the back. Every access moves its node to the front.

---

### 📚 What is a Hash Map?
A **hash map** (Python `dict`) turns a key into a slot instantly, so lookup / insert / delete are **O(1)** on average. Here it lets us jump straight from an `id` to its node instead of scanning a list.

### 📚 What is a Doubly Linked List (DLL)?
A **DLL** is nodes chained both ways (each has `prev` and `next`). Given a node, you can **remove it** or **move it** in **O(1)** — no shifting like an array. Sentinel `head`/`tail` dummy nodes remove edge cases.

---

**Prerequisite knowledge:**
- Dict from key → list node.
- Doubly linked list with dummy head/tail sentinels.

## 📝 Problem

Design a cache with a fixed `capacity` supporting:
- `get(key)` → the value, or `-1` if absent; counts as a **use** (marks it most-recent).
- `put(key, value)` → insert/update; if over capacity, **evict the least-recently-used** entry.

Both operations must run in **O(1)** average time.

**Example**
```
c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
c.get(1)      # -> 1   (1 is now most-recent)
c.put(3, 3)   # evicts key 2 (least-recently-used)
c.get(2)      # -> -1
```

> Two approaches: a naive dict + order-list `O(n)` per op, and hash-map + doubly-linked-list `O(1)`.

### Approach 1 — Dict + Order List (worst)

**Idea:** Store values in a dict and keep a Python list of keys in use-order. On each access, move the key
to the end of the list.

**Time:** **`O(n)` per op** — `list.remove` / `pop(0)` shift elements. **Space:** `O(capacity)`.

In [ ]:
class LRUCacheNaive:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.store = {}                        # key -> value
        self.order = []                        # keys, oldest first ... newest last

    def get(self, key: int) -> int:
        if key not in self.store:
            return -1
        self.order.remove(key)                 # O(n): find and delete the key...
        self.order.append(key)                 # ...then mark it most-recent
        return self.store[key]

    def put(self, key: int, value: int) -> None:
        if key in self.store:
            self.order.remove(key)             # O(n)
        elif len(self.store) >= self.cap:
            oldest = self.order.pop(0)          # O(n): evict the least-recently-used
            del self.store[oldest]
        self.store[key] = value
        self.order.append(key)

### Approach 2 — Hash Map + Doubly Linked List (optimal)

**Idea:** A dict maps `key → node`. Nodes live in a doubly linked list ordered by recency (front = newest).
`get`/`put` unlink the node and re-insert it at the front in O(1); eviction drops the node before the tail.

**Time:** **`O(1)` per op**. **Space:** `O(capacity)`.

In [ ]:
class Node:
    """One entry in the recency list."""
    def __init__(self, key=0, val=0):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}                          # key -> Node
        # Dummy head/tail sentinels remove all the "is it the first/last node?" edge cases.
        self.head = Node()                     # head.next = most-recently-used
        self.tail = Node()                     # tail.prev = least-recently-used
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node: "Node") -> None:
        node.prev.next = node.next             # splice the node out of the list (O(1))
        node.next.prev = node.prev

    def _add_front(self, node: "Node") -> None:
        node.prev = self.head                  # insert right after head = most-recent
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node); self._add_front(node)   # touch -> move to the front
        return node.val

    def put(self, key: int, value: int) -> None:
        if key in self.map:
            self._remove(self.map[key])        # will re-add at the front with the new value
        node = Node(key, value)
        self.map[key] = node
        self._add_front(node)
        if len(self.map) > self.cap:           # over capacity -> evict the LRU (node before tail)
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]

In [ ]:
# Correctness check
def run(CacheClass):
    c = CacheClass(2)
    out = []
    c.put(1, 1); c.put(2, 2)
    out.append(c.get(1))       # 1
    c.put(3, 3)                # evicts key 2
    out.append(c.get(2))       # -1
    c.put(4, 4)                # evicts key 1
    out.append(c.get(1))       # -1
    out.append(c.get(3))       # 3
    out.append(c.get(4))       # 4
    return out

expected = [1, -1, -1, 3, 4]
for CacheClass in (LRUCacheNaive, LRUCache):
    got = run(CacheClass)
    print(CacheClass.__name__, "->", got)
    assert got == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

*(We run a stream of `n` mixed get/put operations; the naive list-based cache is `O(n)` per op → `O(n²)` total.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    cap = max(2, n // 2)
    ops = [('put', i, i) for i in range(n)]          # fill with distinct keys
    ops += [('get', i) for i in range(cap)]          # then touch many existing keys
    return (ops, cap)

def run_ops(ops, cap, CacheClass):
    c = CacheClass(cap); out = []
    for op in ops:
        if op[0] == 'put': c.put(op[1], op[2])
        else: out.append(c.get(op[1]))
    return out
solutions = {
    "dict+list  O(n) per op": lambda ops, cap: run_ops(ops, cap, LRUCacheNaive),
    "hashmap+DLL O(1) per op": lambda ops, cap: run_ops(ops, cap, LRUCache),
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Two structures, two jobs:** hash map for O(1) lookup + doubly linked list for O(1) reorder/evict. Neither alone is enough.
- **Sentinel head/tail:** dummy nodes make every insert/remove uniform — no null checks.
- **Signal:** "O(1) get/set with eviction", "least/most-recently-used", "bounded cache".
- **DevRev / related:** response/session caches, memoizing tool calls (see the Agent project); LeetCode 146 / LFU 460 / 432.
- **Common pitfalls:** (1) using `OrderedDict` in the interview when they asked for the DLL from scratch; (2) forgetting to update the dict on eviction; (3) not moving a node on `get`.